In [ ]:
import os
import glob
import psycopg2
from psycopg2 import sql
from psycopg2.extras import execute_values
from configparser import ConfigParser

def load_config(filename="database.ini", section="postgresql"):
    parser = ConfigParser()
    parser.read(filename)

    # get section, default to postgresql
    config = {}
    if parser.has_section(section):
        params = parser.items(section)
        for param in params:
            config[param[0]] = param[1]
    else:
        raise Exception(
            "Section {0} not found in the {1} file".format(section, filename)
        )

    return config


def load_sql_files(root_dir: str):
    """
    Walk `root_dir`, find all .sql files, sort them by path,
    and return the list of full paths.
    """
    pattern = os.path.join(root_dir, "**", "*.sql")
    # glob with recursive=True
    files = glob.glob(pattern, recursive=True)
    # sort so that prefixes / directory order determine execution order
    files.sort()
    return files

def create_all_tables(sql_root: str):
    """
    Connect to Postgres and run every .sql file under sql_root.
    """
    config = load_config()
    sql_files = load_sql_files(sql_root)

    with psycopg2.connect(**config) as conn:
        with conn.cursor() as cur:
            for path in sql_files:
                print(f"Applying {os.path.relpath(path, sql_root)}...")
                with open(path, 'r') as f:
                    sql_code = f.read()
                cur.execute(sql_code)
        conn.commit()
    print("All tables created successfully.")

def ingest_dataframe(df, table_name, schema='public', conn=None):
    """
    Bulk-insert the Pandas DataFrame `df` into `schema.table_name`.
    Requires column names in df matching the target table.
    """
    # If no existing connection passed, open a new one
    own_conn = False
    if conn is None:
        config = load_config()
        conn = psycopg2.connect(**config)
        own_conn = True

    cols = list(df.columns)
    # Build the INSERT ... VALUES %s template
    insert_sql = sql.SQL("INSERT INTO {schema}.{table} ({fields}) VALUES %s ON CONFLICT DO NOTHING").format(
        schema=sql.Identifier(schema),
        table=sql.Identifier(table_name),
        fields=sql.SQL(', ').join(map(sql.Identifier, cols))
    )

    # Convert df rows to list of tuples
    data = [tuple(x) for x in df.to_numpy()]

    with conn.cursor() as cur:
        # execute_values does very fast bulk inserts
        execute_values(cur, insert_sql.as_string(conn), data)
    if own_conn:
        conn.commit()
        conn.close()
    else:
        conn.commit()



In [8]:
if __name__ == "__main__":
    here = os.path.dirname(__file__)
    sql_root = os.path.abspath(os.path.join(here, '..', 'sql'))
    run_migrations(sql_root)

NameError: name '__file__' is not defined